In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

# ---------------------------------------------------------
# 1. CONFIGURATION & PATHS
# ---------------------------------------------------------
RAW_DATA_DIR = r"C:\Users\salo\ML_final\data\raw\walmart-recruiting-store-sales-forecasting"

# Helper function to read zipped CSVs directly into memory (no temp files)
def read_zipped_csv(filepath):
    with zipfile.ZipFile(filepath, 'r') as zip_ref:
        # Get the name of the csv file inside the zip
        csv_name = zip_ref.namelist() 
        with zip_ref.open(csv_name) as f:
            return pd.read_csv(f)

# ---------------------------------------------------------
# 2. LOAD DATA
# ---------------------------------------------------------
print("Loading data...")

# Load non-zipped stores.csv
stores_path = os.path.join(RAW_DATA_DIR, "stores.csv")
if os.path.exists(stores_path):
    stores = pd.read_csv(stores_path)
else:
    raise FileNotFoundError("stores.csv not found in raw directory.")

# Load zipped train, features, test
train = read_zipped_csv(os.path.join(RAW_DATA_DIR, "train.csv.zip"))
features = read_zipped_csv(os.path.join(RAW_DATA_DIR, "features.csv.zip"))
test = read_zipped_csv(os.path.join(RAW_DATA_DIR, "test.csv.zip"))

print(f"Train shape: {train.shape}")
print(f"Features shape: {features.shape}")
print(f"Stores shape: {stores.shape}")
print(f"Test shape: {test.shape}")

# ---------------------------------------------------------
# 3. MERGE DATASETS
# ---------------------------------------------------------
print("Merging datasets...")

# Merge train with features on Store and Date
df_train_full = train.merge(features, on=['Store', 'Date'], how='left')

# Merge with stores info on Store ID
df_train_full = df_train_full.merge(stores, on='Store', how='left')

# Do the same for test set
df_test_full = test.merge(features, on=['Store', 'Date'], how='left')
df_test_full = df_test_full.merge(stores, on='Store', how='left')

# Convert Date column to datetime
df_train_full['Date'] = pd.to_datetime(df_train_full['Date'])
df_test_full['Date'] = pd.to_datetime(df_test_full['Date'])

# ---------------------------------------------------------
# 4. CHRONOLOGICAL SPLIT (TRAIN / VALIDATION)
# ---------------------------------------------------------
# Split date: Sep 1, 2012
VAL_START_DATE = '2012-09-01'

df_val = df_train_full[df_train_full['Date'] >= VAL_START_DATE].copy()
df_train = df_train_full[df_train_full['Date'] < VAL_START_DATE].copy()

print(f"\nSplit Results:")
print(f"Training Set Shape: {df_train.shape} (Dates: {df_train['Date'].min()} to {df_train['Date'].max()})")
print(f"Validation Set Shape: {df_val.shape} (Dates: {df_val['Date'].min()} to {df_val['Date'].max()})")
print(f"Test Set Shape: {df_test_full.shape} (Dates: {df_test_full['Date'].min()} to {df_test_full['Date'].max()})")

# ---------------------------------------------------------
# 5. FEATURE ENGINEERING FUNCTION
# ---------------------------------------------------------
def create_features(df):
    """
    Adds lag features, rolling stats, and calendar features.
    Returns the modified dataframe.
    """
    df_out = df.copy()
    
    # Sort by Store, Dept, Date to ensure correct grouping
    df_out = df_out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    
    group_cols = ['Store', 'Dept']
    
    # --- Lag Features ---
    # Shift(1) ensures we don't leak current week's sales into features
    df_out['sales_lag_1'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(1)
    df_out['sales_lag_2'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(2)
    df_out['sales_lag_4'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(4)
    df_out['sales_lag_52'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(52) # Yearly seasonality
    
    # --- Rolling Statistics ---
    # Mean of last 4 weeks
    df_out['roll_mean_4'] = df_out.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
    )
    # Std dev of last 4 weeks
    df_out['roll_std_4'] = df_out.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=4, min_periods=1).std()
    )
    # Mean of last 8 weeks
    df_out['roll_mean_8'] = df_out.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=8, min_periods=1).mean()
    )
    
    # --- Calendar Features ---
    df_out['Year'] = df_out['Date'].dt.year
    df_out['Month'] = df_out['Date'].dt.month
    df_out['DayOfWeek'] = df_out['Date'].dt.dayofweek
    df_out['WeekOfYear'] = df_out['Date'].dt.isocalendar().week.astype(int)
    
    # --- Handle Markdowns ---
    markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
    df_out[markdown_cols] = df_out[markdown_cols].fillna(0)
    
    # --- Encode Categoricals ---
    type_map = {'A': 1, 'B': 2, 'C': 3}
    df_out['Type_encoded'] = df_out['Type'].map(type_map).fillna(0).astype(int)
    
    # Ensure IsHoliday is integer
    if 'IsHoliday' in df_out.columns:
        df_out['IsHoliday_int'] = df_out['IsHoliday'].astype(int)
    else:
        # Fallback if column missing due to merge issues
        df_out['IsHoliday_int'] = 0
        
    return df_out

# ---------------------------------------------------------
# 6. APPLY FEATURES TO TRAIN AND VAL
# ---------------------------------------------------------
# IMPORTANT: We must combine Train and Val temporarily to calculate lags 
# that span across the boundary (e.g., val week 1 needs train week N)
print("\nCreating features...")

df_combined = pd.concat([df_train, df_val], ignore_index=True)
df_combined_featured = create_features(df_combined)

# Split back out AFTER feature creation
df_train_final = df_combined_featured[df_combined_featured['Date'] < VAL_START_DATE].copy()
df_val_final = df_combined_featured[df_combined_featured['Date'] >= VAL_START_DATE].copy()

# Also process Test Set for inference later
# Note: For test set, you usually need to append it to the end of historical data 
# to get its initial lags. Here we just prepare it similarly.
df_test_featured = create_features(df_test_full)

print(f"Final Train Shape: {df_train_final.shape}")
print(f"Final Val Shape: {df_val_final.shape}")

# ---------------------------------------------------------
# 7. PREPARE X/Y FOR MODELING
# ---------------------------------------------------------
# Define columns to drop from features
drop_cols = ['Store', 'Dept', 'Id', 'Date', 'Type', 'IsHoliday', 'Weekly_Sales']
# Only drop if they exist
drop_cols = [c for c in drop_cols if c in df_train_final.columns]

X_train = df_train_final.drop(columns=drop_cols)
y_train = df_train_final['Weekly_Sales']

X_val = df_val_final.drop(columns=drop_cols)
y_val = df_val_final['Weekly_Sales']

# Handle any remaining object types (one-hot encode)
object_cols = X_train.select_dtypes(include=['object']).columns
if len(object_cols) > 0:
    print(f"One-hot encoding: {list(object_cols)}")
    X_train = pd.get_dummies(X_train, columns=object_cols)
    X_val = pd.get_dummies(X_val, columns=object_cols)

# Align columns
train_cols = X_train.columns
val_cols = X_val.columns
missing_in_val = set(train_cols) - set(val_cols)
extra_in_val = set(val_cols) - set(train_cols)

for col in missing_in_val:
    X_val[col] = 0
for col in extra_in_val:
    X_val = X_val.drop(columns=[col])

X_val = X_val.reindex(columns=train_cols)

print("\nData Preparation Complete!")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

# Now you can use X_train, y_train, X_val, y_val for modeling!
